# Lab - Validate, Then Retry with the Error

Three tasks on not trusting the values Claude returns. A well-formed answer with a
wrong value does more damage than no answer: it gets stored.

| Task | What you learn |
| --- | --- |
| 1 | Allow null, so a missing value is said out loud instead of invented |
| 2 | Check the values in code, with an exact error message |
| 3 | Retry with that exact error - a bounded number of times |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `FUELIX_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json
import re

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"

# What Claude is told: what an order ID looks like, and never to invent one.
INSTRUCTION = (
    "Extract the return request from this customer email. "
    "Order IDs look like ORD-12345678. If the email gives no "
    "order number, set order_id to null. Never invent one."
)

EMAILS = [
    "Hi, I want to return the blue jacket from order "
    "ORD-12345678. It arrived last week but it is too small. Alex",
    "Hello, the shoes I got are damaged. Please tell me how "
    "to send them back. Thanks, Sam",
    "The headphones from order ORD-7788 stopped working after "
    "two days. I want a return. Kim",
]

---

## Task 1 - Allow null instead of a made-up value

In [ ]:
# ============================================================
# TASK 1 - Allow null instead of a made-up value
# ============================================================
#
# WHAT TO DO
#   Let order_id be a string OR nothing, and send the
#   instruction with every email. Then extract the first two
#   emails.
#
# WHY IT MATTERS
#   Last time the email without an order number came back with
#   order_id "unknown", and the schema accepted it. The first
#   fix is in the schema itself: allow null. Then tell Claude
#   what an order ID looks like, that the field must be null
#   when the email gives none, and never to invent one. That
#   handles the honest case.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The first fix is in the schema itself".
#
# HOW TO DO IT
#   Two blanks:
#
#       "null"         the second type order_id may have
#       INSTRUCTION    the constant from the setup cell
# ============================================================

# BEGIN SOLUTION
SCHEMA = {
    "type": "object",
    "properties": {
        "order_id": {"type": ["string", "null"]},
        "item": {"type": "string"},
        "reason": {"type": "string", "enum": ["too_small", "too_big", "damaged", "other"]},
    },
    "required": ["order_id", "item", "reason"],
    "additionalProperties": False,
}


def extract_once(email):
    response = client.messages.create(
        model=MODEL,
        max_tokens=1000,
        output_config={"effort": "low",
                       "format": {"type": "json_schema", "schema": SCHEMA}},
        messages=[{"role": "user", "content": INSTRUCTION + "\n\n" + email}],
    )
    for block in response.content:
        if block.type == "text":
            return json.loads(block.text)
# SCAFFOLD: SCHEMA = {
# SCAFFOLD:     "type": "object",
# SCAFFOLD:     "properties": {
# SCAFFOLD:         "order_id": {"type": ["string", ...]},   # "null" - a string, or nothing
# SCAFFOLD:         "item": {"type": "string"},
# SCAFFOLD:         "reason": {"type": "string", "enum": ["too_small", "too_big", "damaged", "other"]},
# SCAFFOLD:     },
# SCAFFOLD:     "required": ["order_id", "item", "reason"],
# SCAFFOLD:     "additionalProperties": False,
# SCAFFOLD: }
# SCAFFOLD:
# SCAFFOLD:
# SCAFFOLD: def extract_once(email):
# SCAFFOLD:     response = client.messages.create(
# SCAFFOLD:         model=MODEL,
# SCAFFOLD:         max_tokens=1000,
# SCAFFOLD:         output_config={"effort": "low",
# SCAFFOLD:                        "format": {"type": "json_schema", "schema": SCHEMA}},
# SCAFFOLD:         messages=[{"role": "user", "content": ... + "\n\n" + email}],   # INSTRUCTION
# SCAFFOLD:     )
# SCAFFOLD:     for block in response.content:
# SCAFFOLD:         if block.type == "text":
# SCAFFOLD:             return json.loads(block.text)
# END SOLUTION: replace each ... below with the value named beside it

try:
    results = [extract_once(email) for email in EMAILS[:2]]
except TypeError as exc:   # provided: a blank left above, said plainly
    if "ellipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for data in results:
    print(data)

check("nullable", schema=SCHEMA, results=results)

---

## Task 2 - Check the values in code

In [ ]:
# ============================================================
# TASK 2 - Check the values in code
# ============================================================
#
# WHAT TO DO
#   Finish validate(): it checks order_id against the exact
#   pattern and returns an error message when it does not
#   match - or None when the data is fine.
#
# WHY IT MATTERS
#   Values can be wrong in a way the schema cannot see: a typo,
#   an order number with four digits instead of eight. The
#   schema only knows "a string or null". Code knows the
#   pattern. And the message is exact - which field, which value
#   - because the retry in Task 3 sends it to Claude.
#
# WHERE TO SEE IT IN THE LECTURE
#   "So we add a validation function in code".
#
# HOW TO DO IT
#   One blank: the pattern of a real order ID.
#
#       r"ORD-\d{8}"   ORD- and exactly eight digits
#                      (\d means a digit, {8} means eight of them)
# ============================================================

# BEGIN SOLUTION
def validate(data):
    order_id = data["order_id"]
    if order_id is not None and not re.fullmatch(r"ORD-\d{8}", order_id):
        return f"order_id {order_id!r} is not a valid order ID"
    return None
# SCAFFOLD: def validate(data):
# SCAFFOLD:     order_id = data["order_id"]
# SCAFFOLD:     if order_id is not None and not re.fullmatch(..., order_id):   # r"ORD-\d{8}"
# SCAFFOLD:         return f"order_id {order_id!r} is not a valid order ID"
# SCAFFOLD:     return None
# END SOLUTION: replace the ... below with the value named beside it

SAMPLES = [   # provided: one valid ID, one honest null, one typo
    {"order_id": "ORD-12345678", "item": "jacket", "reason": "too_small"},
    {"order_id": None, "item": "shoes", "reason": "damaged"},
    {"order_id": "ORD-7788", "item": "headphones", "reason": "damaged"},
]
try:
    for sample in SAMPLES:
        print(repr(sample["order_id"]), "->", validate(sample) or "fine")
except TypeError as exc:   # provided: a blank left above, said plainly
    if "ellipsis" in str(exc) or "must be string" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("validate", validate=validate)

---

## Task 3 - Retry with the exact error

In [ ]:
# ============================================================
# TASK 3 - Retry with the exact error
# ============================================================
#
# WHAT TO DO
#   Turn extract into a loop with up to three attempts. Each
#   attempt sends the request, parses the JSON and validates it.
#   If the data is fine, return it. If not, add Claude's answer
#   and your exact error to the conversation, and try again.
#
# WHY IT MATTERS
#   It is tempting to retry by sending the same request again
#   and hoping for a better answer. That is a coin flip. Sending
#   the exact error back turns the retry into a correction:
#   Claude sees what was wrong and fixes that. Watch the third
#   email: rejected once, then null - the email has no valid
#   order number to give.
#
# WHERE TO SEE IT IN THE LECTURE
#   "And we change extract into a loop with up to three
#   attempts".
#
# HOW TO DO IT
#   Three blanks:
#
#       3             the most attempts - always bounded
#       "assistant"   Claude's own answer goes back as its turn
#       error         the exact message validate() returned
# ============================================================

REJECTED = []   # provided: every rejected attempt is noted here


# BEGIN SOLUTION
def extract(email):
    messages = [{"role": "user", "content": INSTRUCTION + "\n\n" + email}]
    for attempt in range(3):
        response = client.messages.create(
            model=MODEL,
            max_tokens=1000,
            output_config={"effort": "low",
                           "format": {"type": "json_schema", "schema": SCHEMA}},
            messages=messages,
        )
        for block in response.content:
            if block.type == "text":
                text = block.text
        data = json.loads(text)
        error = validate(data)
        if error is None:
            return data
        print("  attempt", attempt + 1, "rejected:", error)
        REJECTED.append(error)
        messages.append({"role": "assistant", "content": text})
        messages.append({"role": "user", "content": error + ". Fix it, or use null."})
    return None
# SCAFFOLD: def extract(email):
# SCAFFOLD:     messages = [{"role": "user", "content": INSTRUCTION + "\n\n" + email}]
# SCAFFOLD:     for attempt in range(...):   # 3 - the most attempts
# SCAFFOLD:         response = client.messages.create(
# SCAFFOLD:             model=MODEL,
# SCAFFOLD:             max_tokens=1000,
# SCAFFOLD:             output_config={"effort": "low",
# SCAFFOLD:                            "format": {"type": "json_schema", "schema": SCHEMA}},
# SCAFFOLD:             messages=messages,
# SCAFFOLD:         )
# SCAFFOLD:         for block in response.content:
# SCAFFOLD:             if block.type == "text":
# SCAFFOLD:                 text = block.text
# SCAFFOLD:         data = json.loads(text)
# SCAFFOLD:         error = validate(data)
# SCAFFOLD:         if error is None:
# SCAFFOLD:             return data
# SCAFFOLD:         print("  attempt", attempt + 1, "rejected:", error)
# SCAFFOLD:         REJECTED.append(error)
# SCAFFOLD:         messages.append({"role": ..., "content": text})   # "assistant" - Claude's own answer
# SCAFFOLD:         messages.append({"role": "user", "content": ... + ". Fix it, or use null."})   # error
# SCAFFOLD:     return None
# END SOLUTION: replace each ... below with the value named beside it

try:
    results = []
    for email in EMAILS:
        data = extract(email)
        print(data)
        results.append(data)
except TypeError as exc:   # provided: a blank left above, said plainly
    if "ellipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("retry", results=results, rejected=REJECTED)

---

## Done

Let the schema shape the answer, let your code validate the values, and when validation
fails, send the exact error back and retry a bounded number of times. A retry can only
fix what the source contains: when the email has no valid order number, the right
answer is null.